# TotalSegmentator Lung Mask

Runs TotalSegmentator on one LIDC scan and saves the final mask overlay as a PNG.

**Requirements:** `pip install TotalSegmentator nibabel`

**Output:** `ts_mask_<patient_id>_slice<N>.png` in this directory.

In [17]:
%matplotlib inline
import os, tempfile, shutil
import pylidc as pl
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label, binary_fill_holes, binary_dilation, center_of_mass
from skimage.morphology import disk as morpho_disk

try:
    import nibabel as nib
    from totalsegmentator.python_api import totalsegmentator as run_ts
    import torch
    TS_AVAILABLE = True
    print('TotalSegmentator available')
except ImportError:
    TS_AVAILABLE = False
    print('ERROR: run  pip install TotalSegmentator nibabel')

HU_MIN, HU_MAX     = -1000, 400
HU_THRESHOLD       = -500
MASK_DILATION      = 3
TS_LUNG_LABELS = [
    'lung_upper_lobe_left', 'lung_lower_lobe_left',
    'lung_upper_lobe_right', 'lung_middle_lobe_right',
    'lung_lower_lobe_right',
]
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('ts_mask.ipynb'))


def get_hu_mask(raw_slice):
    """Threshold -> label -> border removal -> 2 largest -> combine -> disk(3)."""
    binary = raw_slice < HU_THRESHOLD
    labeled, _ = label(binary)
    border = set()
    border.update(labeled[0, :]); border.update(labeled[-1, :])
    border.update(labeled[:, 0]); border.update(labeled[:, -1])
    border.discard(0)
    for b in border:
        labeled[labeled == b] = 0
    labeled2, n = label(labeled > 0)
    if n == 0:
        return np.zeros(raw_slice.shape, dtype='uint8')
    sizes = np.bincount(labeled2.ravel())[1:]
    keep  = np.argsort(sizes)[::-1][:min(2, n)] + 1
    regions = [labeled2 == k for k in keep]
    if len(regions) == 2:
        cx = [center_of_mass(m)[1] for m in regions]
        regions = [regions[int(np.argmin(cx))], regions[int(np.argmax(cx))]]
    combined = regions[0].copy()
    for r in regions[1:]:
        combined |= r
    return binary_dilation(combined, structure=morpho_disk(3)).astype('uint8')


In [18]:
scan = pl.query(pl.Scan).first()
vol  = scan.to_volume()
slice_idx = vol.shape[2] // 2
raw_slice  = vol[:, :, slice_idx].astype('float32')
display    = (np.clip(raw_slice, HU_MIN, HU_MAX) - HU_MIN) / (HU_MAX - HU_MIN)

hu_mask_2d = get_hu_mask(raw_slice)

print(f'Patient:         {scan.patient_id}')
print(f'Volume shape:    {vol.shape}')
print(f'Slice:           {slice_idx}')
print(f'HU mask pixels:  {hu_mask_2d.sum()}')


In [19]:
assert TS_AVAILABLE, 'TotalSegmentator not installed'

device  = 'gpu' if torch.cuda.is_available() else 'cpu'
tmpdir  = tempfile.mkdtemp(prefix=f'ts_{scan.patient_id}_')
nifti_in  = os.path.join(tmpdir, 'ct.nii.gz')
nifti_out = os.path.join(tmpdir, 'seg')
os.makedirs(nifti_out, exist_ok=True)

px, sl = float(scan.pixel_spacing), float(scan.slice_thickness)
nib.save(nib.Nifti1Image(vol.astype('float32'), np.diag([px, px, sl, 1.0])), nifti_in)

print(f'Running TotalSegmentator on {device} ...')
run_ts(nifti_in, nifti_out, task='total', device=device, fast=True,
       quiet=True, roi_subset=TS_LUNG_LABELS)
print('Done.')


In [20]:
combined = None
for lname in TS_LUNG_LABELS:
    seg_path = os.path.join(nifti_out, f'{lname}.nii.gz')
    if os.path.exists(seg_path):
        seg = nib.load(seg_path).get_fdata()
        combined = seg if combined is None else combined + seg

assert combined is not None, 'No lobe segments found'

ts_3d = binary_fill_holes((combined > 0).astype(bool))
ts_3d = binary_dilation(ts_3d, iterations=MASK_DILATION)
ts_mask_2d = ts_3d[:, :, slice_idx].astype('uint8')

print(f'TS mask pixels: {ts_mask_2d.sum()} / {ts_mask_2d.size} '
      f'({100*ts_mask_2d.sum()/ts_mask_2d.size:.1f}%)')

# Save PNG
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(display, cmap='gray')
axes[0].set_title(f'CT slice {slice_idx} -- {scan.patient_id}')
axes[0].axis('off')
axes[1].imshow(ts_mask_2d, cmap='gray')
axes[1].set_title(f'TS mask ({ts_mask_2d.sum()} px)')
axes[1].axis('off')
axes[2].imshow(display, cmap='gray')
axes[2].imshow(ts_mask_2d, cmap='Blues', alpha=0.45)
axes[2].set_title('TS mask overlay')
axes[2].axis('off')
plt.suptitle(f'{scan.patient_id} -- slice {slice_idx}', fontsize=13)
plt.tight_layout()

out_path = os.path.join(NOTEBOOK_DIR, f'ts_mask_{scan.patient_id}_slice{slice_idx}.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')


In [ ]:
hu  = hu_mask_2d.astype(bool)
ts  = ts_mask_2d.astype(bool)

intersection = (hu & ts).sum()
union        = (hu | ts).sum()
iou  = intersection / union             if union             > 0 else float('nan')
dice = 2*intersection / (hu.sum() + ts.sum()) if (hu.sum() + ts.sum()) > 0 else float('nan')

# Colour diff map: red=HU only, blue=TS only, white=both
diff = np.zeros((*hu.shape, 3), dtype='float32')
diff[hu & ~ts] = [1, 0, 0]
diff[ts & ~hu] = [0, 0, 1]
diff[hu &  ts] = [1, 1, 1]

stats = '\n'.join([
    'Mask similarity', '',
    f'HU pixels : {hu.sum()}',
    f'TS pixels : {ts.sum()}',
    f'Overlap   : {intersection}',
    f'Union     : {union}', '',
    f'IoU  = {iou:.4f}',
    f'Dice = {dice:.4f}',
])

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(display, cmap='gray')
axes[0, 0].set_title(f'CT slice {slice_idx}')
axes[0, 0].axis('off')

axes[0, 1].imshow(display, cmap='gray')
axes[0, 1].imshow(hu_mask_2d, cmap='Greens', alpha=0.45)
axes[0, 1].set_title(f'HU mask  ({hu.sum()} px)')
axes[0, 1].axis('off')

axes[0, 2].imshow(display, cmap='gray')
axes[0, 2].imshow(ts_mask_2d, cmap='Blues', alpha=0.45)
axes[0, 2].set_title(f'TS mask  ({ts.sum()} px)')
axes[0, 2].axis('off')

axes[1, 0].imshow(diff)
axes[1, 0].set_title('Diff: red=HU only  blue=TS only  white=both')
axes[1, 0].axis('off')

axes[1, 1].imshow(display, cmap='gray')
axes[1, 1].contour(hu_mask_2d, levels=[0.5], colors='lime', linewidths=1.2)
axes[1, 1].contour(ts_mask_2d, levels=[0.5], colors='cyan', linewidths=1.2)
axes[1, 1].set_title('Contours: lime=HU  cyan=TS')
axes[1, 1].axis('off')

axes[1, 2].axis('off')
axes[1, 2].text(
    0.5, 0.5, stats,
    ha='center', va='center', fontsize=13,
    transform=axes[1, 2].transAxes,
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85),
)

plt.suptitle(f'{scan.patient_id} -- slice {slice_idx} -- IoU {iou:.3f}  Dice {dice:.3f}', fontsize=13)
plt.tight_layout()

cmp_path = os.path.join(NOTEBOOK_DIR, f'mask_compare_{scan.patient_id}_slice{slice_idx}.png')
plt.savefig(cmp_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'IoU  = {iou:.4f}')
print(f'Dice = {dice:.4f}')
print(f'Saved: {cmp_path}')


SyntaxError: unterminated string literal (detected at line 15) (403867827.py, line 15)

In [ ]:
shutil.rmtree(tmpdir, ignore_errors=True)
print('Cleaned up temp dir')
